# RSNA Knee Abnormality Detection: Custom RadImageNet Head Training
### Training on 0.893 AUC Consensus LLM Pseudo-Labels with Confidence Masking

**Key Pipeline Highlights**:
1. **Supervision**: High-quality `llm_labels_v4_blend.csv` (0.893 AUC vs gold, blending Steven Lee Hans + Lixin73 GPT-5.6-Sol).
2. **Confidence-Weighted Masked Loss**: $w = 2 \cdot |p - 0.5|$ ignores unaddressed radiology findings ($p=0.5 \rightarrow w=0$) and learns from confident signals.
3. **Medical Backbone**: Frozen RadImageNet ResNet-50 (`ResNet50.pt`) pretrained on 1.35M clinical examinations.
4. **Proven 0.920 DICOM Extraction**: 130 mm physical crop, `InstanceNumber` slice ordering, left/right knee alignment.
5. **Validation**: 5-Fold Cross-Validation evaluated strictly on held-out gold expert studies (58 ground truth cases).
6. **Output**: 5 trained fold checkpoints (`custom_rad_head_fold0..4.pt`) ready to blend into the 0.922 SOTA ensemble.

In [ ]:
import os
import gc
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
cv2.setNumThreads(1)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEV}')
if DEV.type == 'cuda':
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')

# Seed for reproducibility
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]
print(f'Configured {len(TARGETS)} targets: {TARGETS}')

In [ ]:
# Dynamic Input Discovery
INPUT_DIR = Path('/kaggle/input')

# 1. Competition Directory
COMP_DIR = None
for d in [INPUT_DIR / 'rsna-knee-abnormality-detection', INPUT_DIR / 'competitions' / 'rsna-knee-abnormality-detection']:
    if d.exists():
        COMP_DIR = d
        break
if COMP_DIR is None:
    # Search fallback
    matches = list(INPUT_DIR.glob('*rsna-knee-abnormality-detection*'))
    if matches:
        COMP_DIR = matches[0]
assert COMP_DIR is not None, 'Could not find RSNA Knee Abnormality Detection competition directory in /kaggle/input!'
print(f'Found Competition Data: {COMP_DIR}')

# 2. RadImageNet ResNet-50 Backbone
BACKBONE_PATH = None
for p in INPUT_DIR.rglob('*ResNet50.pt'):
    if p.is_file():
        BACKBONE_PATH = p
        break
if BACKBONE_PATH is None:
    for p in INPUT_DIR.rglob('*.pt*'):
        if 'rad' in str(p).lower() and 'resnet' in str(p).lower():
            BACKBONE_PATH = p
            break
assert BACKBONE_PATH is not None, 'Could not find RadImageNet ResNet50.pt! Please attach resnet-50-radimagenet-marwan dataset.'
print(f'Found RadImageNet Weights: {BACKBONE_PATH}')

# 3. LLM Consensus Labels (llm_labels_v4_blend.csv or similar)
LABEL_CSV = None
for p in INPUT_DIR.rglob('*llm_labels_v4_blend.csv'):
    if p.is_file():
        LABEL_CSV = p
        break
if LABEL_CSV is None:
    for p in INPUT_DIR.rglob('*llm_labels*.csv'):
        if p.is_file() and 'v4' in p.name.lower():
            LABEL_CSV = p
            break
if LABEL_CSV is None:
    for p in INPUT_DIR.rglob('*llm_labels*.csv'):
        if p.is_file():
            LABEL_CSV = p
            break
assert LABEL_CSV is not None, 'Could not find llm_labels_v4_blend.csv! Please attach your rsna-llm-labels dataset.'
print(f'Found Consensus Labels: {LABEL_CSV}')

In [ ]:
# Load DataFrames
train_df = pd.read_csv(COMP_DIR / 'train.csv')
series_df = pd.read_csv(COMP_DIR / 'train_series.csv')
pseudo_df = pd.read_csv(LABEL_CSV)

print(f'Total Train Studies: {len(train_df)}')
print(f'Total Series Entries: {len(series_df)}')
print(f'Total Pseudo-Label Studies: {len(pseudo_df)}')

# Extract 58 Expert Gold-Annotated Studies
gold_df = train_df[train_df[TARGETS].notna().all(axis=1)].copy().reset_index(drop=True)
print(f'Expert Gold Studies Count: {len(gold_df)}')
assert len(gold_df) == 58, f'Expected 58 gold studies, found {len(gold_df)}'

# 5-Fold Stratified Split across Gold Studies
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
gold_df['fold'] = -1
for fold, (_, val_idx) in enumerate(kf.split(gold_df)):
    gold_df.loc[val_idx, 'fold'] = fold
print('Gold Studies per fold:', gold_df['fold'].value_counts().to_dict())

# Filter pseudo-labeled studies: exclude gold studies from pseudo candidate pool to keep pure validation
gold_uids = set(gold_df['StudyInstanceUID'])
pseudo_clean = pseudo_df[~pseudo_df['StudyInstanceUID'].isin(gold_uids)].copy().reset_index(drop=True)
print(f'Candidate Pseudo Studies available for training: {len(pseudo_clean)}')

# Pre-index series metadata by StudyInstanceUID for O(1) lookup
series_by_study = {}
for r in series_df.to_dict('records'):
    s_uid = str(r['StudyInstanceUID'])
    series_by_study.setdefault(s_uid, []).append(r)
print(f'Indexed series metadata for {len(series_by_study)} studies')

In [ ]:
# DICOM extraction functions (derived from proven 0.920 SOTA pipeline)
CROP_MM = 130.0
SIZE = 224
N_SLICE = 6
SLICE_BAND = (0.2, 0.8)
SLOTS = [
    ('Sagittal', 1), ('Sagittal', 0),
    ('Coronal', 1),  ('Coronal', 0),
    ('Axial', 1),    ('Axial', 0)
]
N_SLOT = len(SLOTS)
SERIES_ROOT = COMP_DIR / 'train_series'

def ordered_files(sdir, cap=64):
    """Fast DICOM header ordering by InstanceNumber in <1ms."""
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(study, recs):
    """Extracts standardized (N_SLOT, N_SLICE, SIZE, SIZE) tensor."""
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            s_uid = str(sub.iloc[0]['SeriesInstanceUID'])
            files = ordered_files(SERIES_ROOT / str(study) / s_uid)
            if not files:
                continue
            flip = (plane != 'Sagittal') and (series_side(files[0]) < 0)
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            for c, p in enumerate(picks):
                img = render(files[p], flip)
                if img is None and p + 1 < len(files):
                    img = render(files[p + 1], flip)
                if img is not None:
                    out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return out, mask

print('DICOM extraction engine ready.')

In [ ]:
# Load Pretrained RadImageNet ResNet-50 Encoder
class RadEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])

    def forward(self, x):
        return self.backbone(x).mean(dim=(2, 3))

encoder = RadEncoder()
state = torch.load(BACKBONE_PATH, map_location='cpu', weights_only=True)
# Handle both raw state_dict and nested checkpoint formats
if 'state_dict' in state:
    state = state['state_dict']
elif 'model' in state:
    state = state['model']
# Clean up prefix if needed
clean_state = {}
for k, v in state.items():
    k_clean = k.replace('module.', '')
    if not k_clean.startswith('backbone.'):
        k_clean = f'backbone.{k_clean}'
    clean_state[k_clean] = v

encoder.load_state_dict(clean_state, strict=False)
encoder.eval().to(DEV)
for p in encoder.parameters():
    p.requires_grad = False
print(f'RadImageNet ResNet-50 Encoder Loaded: {sum(p.numel() for p in encoder.parameters()):,} params (FROZEN)')

# Configure Training Study Pool
# Top 1,000 highest-confidence pseudo studies + all 58 gold studies
# (Confidence metric: distance from 0.5 across the 12 findings)
pseudo_clean['conf_score'] = pseudo_clean[TARGETS].apply(lambda row: np.mean(2 * np.abs(row.values - 0.5)), axis=1)
pseudo_clean = pseudo_clean.sort_values('conf_score', ascending=False).reset_index(drop=True)

N_PSEUDO_TRAIN = 1000  # 1,000 top confident studies for fast ~20 min feature extraction
selected_pseudo = pseudo_clean.head(N_PSEUDO_TRAIN).copy()
train_pool_uids = list(gold_df['StudyInstanceUID']) + list(selected_pseudo['StudyInstanceUID'])
print(f'Total studies in training pool: {len(train_pool_uids)} (58 Gold + {len(selected_pseudo)} High-Confidence Pseudo)')

# Feature Extraction Pass
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

feats = {}
t_extract_start = time.time()
print('Starting RadImageNet Feature Extraction...')

with torch.no_grad():
    for study in tqdm(train_pool_uids, desc='Extracting Features'):
        recs = series_by_study.get(str(study), [])
        if not recs:
            continue
        imgs, mask = build_study(study, recs)
        if mask.sum() == 0:
            continue
        flat = imgs.reshape(-1, SIZE, SIZE)
        populated = np.repeat(mask, N_SLICE) > 0
        flat = flat[populated]
        if len(flat) == 0:
            continue
        x = torch.from_numpy(flat).float().div(255.0).unsqueeze(1).repeat(1, 3, 1, 1).to(DEV)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        f = encoder(x).mean(dim=0)  # Mean-pool across all populated slices -> (2048,)
        feats[str(study)] = f.cpu()
        del x, f
    torch.cuda.empty_cache()

elapsed = time.time() - t_extract_start
print(f'Successfully extracted and cached 2048-dim features for {len(feats)}/{len(train_pool_uids)} studies in {elapsed/60:.1f} minutes!')

In [ ]:
# Trainable Head Architecture
class RadAttentionHead(nn.Module):
    def __init__(self, in_dim=2048, hidden=512, n_out=len(TARGETS), dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_out)
        )

    def forward(self, x):
        return self.net(x)

def confidence_masked_bce(logits, targets, weights):
    """
    Binary Cross Entropy with per-cell confidence weighting:
    weight = 2 * |p - 0.5|
    """
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    weighted_loss = loss * weights
    denom = weights.sum().clamp_min(1.0)
    return weighted_loss.sum() / denom

# Build Training and Validation Datasets
usable_gold = gold_df[gold_df['StudyInstanceUID'].astype(str).isin(feats.keys())].copy().reset_index(drop=True)
usable_pseudo = selected_pseudo[selected_pseudo['StudyInstanceUID'].astype(str).isin(feats.keys())].copy().reset_index(drop=True)

print(f'Usable Gold Studies: {len(usable_gold)} (Target: 58)')
print(f'Usable Pseudo Studies: {len(usable_pseudo)}')

# Prepare Target Matrices and Confidence Weights
# Pseudo labels: targets in [0, 1], weights = 2 * |p - 0.5|
pseudo_targets = usable_pseudo[TARGETS].values.astype(np.float32)
pseudo_weights = (2.0 * np.abs(pseudo_targets - 0.5)).astype(np.float32)

# Gold labels: targets in {0, 1}, weights = 1.0 unconditionally
gold_targets = usable_gold[TARGETS].values.astype(np.float32)
gold_weights = np.ones_like(gold_targets, dtype=np.float32)

print('Target & Weight matrices ready.')

In [ ]:
# 5-Fold Training Loop
OUTPUT_DIR = Path('/kaggle/working')
N_EPOCHS = 35
LR = 1e-3
WEIGHT_DECAY = 1e-2

fold_aucs = []
oof_preds = np.zeros_like(gold_targets)

for fold in range(N_FOLDS):
    print(f'\n================== Fold {fold} ==================')
    val_gold_mask = (usable_gold['fold'] == fold).values
    train_gold_mask = ~val_gold_mask

    # Validation: Held-out gold studies ONLY
    X_val = torch.stack([feats[str(s)] for s in usable_gold[val_gold_mask]['StudyInstanceUID']]).to(DEV)
    Y_val = torch.from_numpy(gold_targets[val_gold_mask]).to(DEV)

    # Training: Training gold studies + all usable pseudo studies
    X_train_gold = torch.stack([feats[str(s)] for s in usable_gold[train_gold_mask]['StudyInstanceUID']])
    X_train_pseudo = torch.stack([feats[str(s)] for s in usable_pseudo['StudyInstanceUID']])
    X_train = torch.cat([X_train_gold, X_train_pseudo], dim=0).to(DEV)

    Y_train_gold = torch.from_numpy(gold_targets[train_gold_mask])
    Y_train_pseudo = torch.from_numpy(pseudo_targets)
    Y_train = torch.cat([Y_train_gold, Y_train_pseudo], dim=0).to(DEV)

    W_train_gold = torch.from_numpy(gold_weights[train_gold_mask])
    W_train_pseudo = torch.from_numpy(pseudo_weights)
    W_train = torch.cat([W_train_gold, W_train_pseudo], dim=0).to(DEV)

    # Initialize Head & Optimizer
    model = RadAttentionHead().to(DEV)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-5)

    best_val_auc = 0.0
    best_probs = None
    best_weights = None

    # Epoch Training Loop (fast execution on cached tensors)
    batch_size = 64
    n_samples = X_train.shape[0]

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_samples)
        train_loss = 0.0
        n_batches = 0

        for b_start in range(0, n_samples, batch_size):
            b_idx = perm[b_start:b_start + batch_size]
            xb, yb, wb = X_train[b_idx], Y_train[b_idx], W_train[b_idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = confidence_masked_bce(logits, yb, wb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            n_batches += 1

        scheduler.step()
        train_loss /= max(1, n_batches)

        # Validation on Gold Studies
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
            val_true = Y_val.cpu().numpy()

            # Calculate macro AUC over valid columns
            col_aucs = []
            for c in range(len(TARGETS)):
                if len(np.unique(val_true[:, c])) > 1:
                    try:
                        col_aucs.append(roc_auc_score(val_true[:, c], val_probs[:, c]))
                    except Exception:
                        pass
            macro_auc = np.mean(col_aucs) if col_aucs else 0.5

        if macro_auc > best_val_auc:
            best_val_auc = macro_auc
            best_probs = val_probs
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == N_EPOCHS:
            print(f'  Epoch {epoch:02d}/{N_EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Gold AUC: {macro_auc:.4f} (Best: {best_val_auc:.4f})')

    # Save Best Checkpoint for this Fold
    ckpt_path = OUTPUT_DIR / f'custom_rad_head_fold{fold}.pt'
    torch.save(best_weights, ckpt_path)
    print(f'>> Saved Fold {fold} Best Checkpoint: {ckpt_path} (AUC: {best_val_auc:.4f})')
    fold_aucs.append(best_val_auc)
    oof_preds[val_gold_mask] = best_probs

print(f'\n=============================================')
print(f'5-Fold Mean Gold Validation AUC: {np.mean(fold_aucs):.4f}')
for f_idx, a in enumerate(fold_aucs):
    print(f'  Fold {f_idx}: {a:.4f}')

In [ ]:
# Overall Out-Of-Fold Validation AUC
overall_col_aucs = {}
for c, target in enumerate(TARGETS):
    try:
        score = roc_auc_score(gold_targets[:, c], oof_preds[:, c])
        overall_col_aucs[target] = score
    except Exception:
        overall_col_aucs[target] = 0.5

print('Overall Gold OOF AUC per Finding:')
for target, auc_val in overall_col_aucs.items():
    print(f'  - {target:<20}: {auc_val:.4f}')
print(f'\nOverall Macro-Averaged OOF Gold AUC: {np.mean(list(overall_col_aucs.values())):.4f}')

# Save OOF Predictions DataFrame
oof_df = pd.DataFrame(oof_preds, columns=TARGETS)
oof_df.insert(0, 'StudyInstanceUID', usable_gold['StudyInstanceUID'])
oof_df.to_csv(OUTPUT_DIR / 'oof_predictions.csv', index=False)
print(f'Saved OOF Predictions to {OUTPUT_DIR / "oof_predictions.csv"}')

print('\nAll 5 fold checkpoints generated and ready to download/dataset:')
for f in sorted(OUTPUT_DIR.glob('custom_rad_head_fold*.pt')):
    print(f'  * {f.name} ({f.stat().st_size / 1024:.1f} KB)')